# PSN-2 Training — Kaggle

**Setup:** Attach `psn2_kaggle_full.zip` as a dataset before running.

**Session plan (PRD Section 17.3):**
- Session 1-2: D1 — ARC-AGI-2 grids + synthetic relational graphs
- Session 3-4: D2 — Causal grounding
- Session 5:   D3 — ToM/ToMi theory-of-mind
- Session 6-7: D4 — Wikitext linguistic grounding
- Session 8-9: D5 — ARC-AGI-2 + GSM8K abstract reasoning
- Session 10+: D6 — Full integration + BBH

Each session auto-resumes from the previous checkpoint.

In [ ]:
import os, subprocess, sys, json

# ── Locate the archive ──────────────────────────────────────────────────────
ARCHIVE = None
for root in ['/kaggle/input', '/kaggle/working', '.']:
    for dirpath, _, files in os.walk(root):
        for f in files:
            if f in ('psn2_kaggle_full.zip', 'psn2_kaggle_full.tar.gz'):
                ARCHIVE = os.path.join(dirpath, f)
                break
        if ARCHIVE: break
    if ARCHIVE: break

assert ARCHIVE, 'psn2_kaggle_full.zip not found — attach it as a dataset'
print('Found archive:', ARCHIVE)

In [ ]:
import zipfile, tarfile
WORKDIR = '/kaggle/working/psn2_repo'
os.makedirs(WORKDIR, exist_ok=True)

if ARCHIVE.endswith('.zip'):
    with zipfile.ZipFile(ARCHIVE) as zf:
        zf.extractall(WORKDIR)
else:
    with tarfile.open(ARCHIVE) as tf:
        tf.extractall(WORKDIR)

sys.path.insert(0, WORKDIR)
os.chdir(WORKDIR)
print('Extracted to', WORKDIR)
print('Contents:', sorted(os.listdir(WORKDIR)))

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tqdm'], check=True)

import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── Verify data is present ──────────────────────────────────────────────────
data_dir = os.path.join(WORKDIR, 'data')
expected = [
    'd5_arc_agi2/train.jsonl',
    'd3_tom/train.jsonl',
    'd3_tomi/train.jsonl',
    'd4_wikitext/train.jsonl',
    'd5_gsm8k/train.jsonl',
    'd6_bbh/test.jsonl',
]
for rel in expected:
    p = os.path.join(data_dir, rel)
    exists = os.path.exists(p)
    size_mb = os.path.getsize(p) / 1e6 if exists else 0
    status = f'OK ({size_mb:.1f} MB)' if exists else 'MISSING'
    print(f'  {rel:35s} {status}')

In [ ]:
# ── Configure for this session ──────────────────────────────────────────────
# EDIT THIS: set the stage for this Kaggle session
STAGE = 'D1'   # D1 | D2 | D3 | D4 | D5 | D6

# Steps per session (9h budget at ~1 step/sec for D1 = ~32k steps)
STEPS_PER_STAGE = {
    'D1': 20000,
    'D2': 20000,
    'D3': 15000,
    'D4': 25000,
    'D5': 30000,
    'D6': 40000,
}

cfg_path = os.path.join(WORKDIR, 'configs/default.json')
with open(cfg_path) as f:
    cfg = json.load(f)

cfg.update({
    'stage':            STAGE,
    'vsa_dim':          512,
    'max_nodes':        256,
    'grid_size':        8,
    'grid_vocab':       10,
    'rel_vocab_size':   64,
    'batch_size':       32,
    'steps':            STEPS_PER_STAGE[STAGE],
    'lr_ff':            1e-4,
    'log_every':        200,
    'checkpoint_every': 1000,
    'checkpoint_dir':   '/kaggle/working/artifacts',
    'data_dir':         data_dir,
    'max_wikitext_samples': 50000,
    'n_synthetic_samples':  5000,
})

with open(cfg_path, 'w') as f:
    json.dump(cfg, f, indent=2)

print(f'Stage: {STAGE} | Steps: {cfg["steps"]} | Batch: {cfg["batch_size"]}')

In [ ]:
# ── Smoke test before training ──────────────────────────────────────────────
result = subprocess.run(
    [sys.executable, 'scripts/smoke_test.py'],
    cwd=WORKDIR, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('SMOKE TEST FAILED:')
    print(result.stderr)
else:
    print('Smoke test passed.')

In [ ]:
# ── Training ─────────────────────────────────────────────────────────────────
# Auto-resume from previous session checkpoint if it exists
latest_ckpt = '/kaggle/working/artifacts/latest.pt'

# Check for checkpoint from a previous Kaggle session (attached as dataset)
prev_ckpt = None
for root in ['/kaggle/input']:
    for dirpath, _, files in os.walk(root):
        if 'latest.pt' in files:
            prev_ckpt = os.path.join(dirpath, 'latest.pt')
            break

# Copy previous checkpoint to working dir if present
if prev_ckpt and not os.path.exists(latest_ckpt):
    import shutil
    os.makedirs('/kaggle/working/artifacts', exist_ok=True)
    shutil.copy(prev_ckpt, latest_ckpt)
    print(f'Loaded previous checkpoint from {prev_ckpt}')

resume_flag = ['--resume', latest_ckpt] if os.path.exists(latest_ckpt) else []
cmd = [sys.executable, 'train.py', '--config', cfg_path] + resume_flag
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, cwd=WORKDIR)
print('Exit code:', result.returncode)

In [ ]:
# ── Evaluation ───────────────────────────────────────────────────────────────
if os.path.exists(latest_ckpt):
    result = subprocess.run([
        sys.executable, 'evaluate.py',
        '--config', cfg_path,
        '--checkpoint', latest_ckpt,
        '--output', '/kaggle/working/eval_results.json',
    ], cwd=WORKDIR)
    print('Eval exit code:', result.returncode)
    if os.path.exists('/kaggle/working/eval_results.json'):
        with open('/kaggle/working/eval_results.json') as f:
            results = json.load(f)
        print(json.dumps(results, indent=2))
else:
    print('No checkpoint found')

In [ ]:
# ── List output artifacts ─────────────────────────────────────────────────────
artifacts_dir = '/kaggle/working/artifacts'
if os.path.exists(artifacts_dir):
    for f in sorted(os.listdir(artifacts_dir)):
        size_mb = os.path.getsize(os.path.join(artifacts_dir, f)) / 1e6
        print(f'  {f:45s}  {size_mb:.1f} MB')
else:
    print('No artifacts yet')